In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/gemma/keras/gemma_instruct_2b_en/2/config.json
/kaggle/input/gemma/keras/gemma_instruct_2b_en/2/tokenizer.json
/kaggle/input/gemma/keras/gemma_instruct_2b_en/2/metadata.json
/kaggle/input/gemma/keras/gemma_instruct_2b_en/2/model.weights.h5
/kaggle/input/gemma/keras/gemma_instruct_2b_en/2/assets/tokenizer/vocabulary.spm
/kaggle/input/the-penal-code-1860/THE PENAL CODE 1860.pdf


In [2]:
import os
import requests

# Get PDF document path
pdf_path = "/kaggle/input/the-penal-code-1860/THE PENAL CODE 1860.pdf"

In [3]:
!pip install PyMuPDF
!pip install tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 85.7 MB/s eta 0:00:00:00:0100:01


In [4]:
import fitz #for opening document
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
   
    cleaned_text = text.replace("\n", " ").strip()
    return cleaned_text

def open_and_read_pdf(pdf_path: str) -> list[dict]:
   
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):
        text = page.get_text()
        text = text_formatter(text=text)
        pages_and_texts.append({"page_number": page_number - 41, # adjusted page numbers since our PDF starts on page 42
                                "page_char_count": len(text),
                                "page_word_count": len(text.split(" ")),
                                "page_sentence_count_raw": len(text.split(", ")),
                                "page_token_count": len(text) / 4, #1 token has approx 4 charactersz
                                "text": text})
    return pages_and_texts
pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': -41,
  'page_char_count': 1967,
  'page_word_count': 503,
  'page_sentence_count_raw': 14,
  'page_token_count': 491.75,
  'text': '1 THE PENAL CODE, 1860    (ACT NO. XLV OF 1860).        [6th October, 1860]                             CHAPTER I    INTRODUCTION              Preamble     WHEREAS it is expedient to provide a general Penal Code for Bangladesh; It is  enacted as follows:-                         Title and extent of  operation of the  Code     1. This Act shall be called the 2[ Penal Code], and shall take effect throughout  Bangladesh.                         Punishment of  offences committed  within Bangladesh     2. Every person shall be liable to punishment under this Code and not otherwise for  every act or omission contrary to the provisions thereof, of which he shall be guilty  within Bangladesh.                         Punishment of  offences committed  beyond, but which  by law may be tried  within Bangladesh     3. Any person liable, by any Banglad

In [5]:
import random

random.sample(pages_and_texts, k=2)

[{'page_number': -40,
  'page_char_count': 2697,
  'page_word_count': 647,
  'page_sentence_count_raw': 25,
  'page_token_count': 674.25,
  'text': '(b) B, a European British subject, commits a murder in 3[ Rangpur]. He can be tried  and convicted of murder in any place in Bangladesh in which he may be found.    (c) C, a foreigner who is in the service of the Bangladesh Government, commits a  murder in 4[ Khulna]. He can be tried and convicted of murder at any place in  Bangladesh in which he may be found.    (d) D, a British subject living in 5[ Khulna], instigates E to commit a murder in 6[  Chittagong]. D is guilty of abetting murder.                         Certain laws not to  be affected by this  Act     5. Nothing in this Act is intended to repeal, vary, suspend, or affect 7[ * * *] any of the  provisions of any Act for punishing mutiny and desertion of officers, soldiers, sailors or  airmen in the service of the 8[ Republic], or of any special or local law.                CHAPT

### Get some stats on the text

Let's perform a rough exploratory data analysis (EDA) to get an idea of the size of the texts (e.g. character counts, word counts etc) we're working with.




In [6]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,1967,503,14,491.75,"1 THE PENAL CODE, 1860 (ACT NO. XLV OF 1860..."
1,-40,2697,647,25,674.25,"(b) B, a European British subject, commits a m..."
2,-39,2150,575,25,537.50,“Person” 11. The word “person” includes an...
3,-38,3007,594,51,751.75,"“Court of justice” 20. The words ""Court of..."
4,-37,2533,618,16,633.25,A Municipal Commissioner is a public servant. ...


In [7]:
df.tail()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
16,-25,3361,766,56,840.25,or drink intended for sale noxious as food o...
17,-24,2968,629,59,742.00,"has in his possession any obscene book, pamphl..."
18,-23,3496,750,55,874.00,"Offering of prize in connection with trade, ..."
19,-22,3184,698,31,796.00,Of Criminal Force and Assault Punishment fo...
20,-21,958,191,16,239.50,Assault or criminal force on grave provocati...


In [8]:
df.shape

(21, 6)

In [9]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,21.0,21.00,21.00,21.00,21.00
mean,-31.0,2818.05,640.76,38.24,704.51
std,6.2,610.33,126.45,15.41,152.58
min,-41.0,958.00,191.00,14.00,239.50
25%,-36.0,2609.00,599.00,25.00,652.25
50%,-31.0,2968.00,683.00,41.00,742.00
75%,-26.0,3204.00,711.00,51.00,801.00
max,-21.0,3572.00,766.00,62.00,893.00


### Further text processing (splitting pages into sentences)
We will to follow the workflow of:

`Ingest text -> split it into groups/chunks -> embed the groups/chunks -> use the embeddings`

Why split into sentences?

* Easier to handle than larger pages of text (especially if pages are densely filled with text).
* Can get specific and find out which group of sentences were used to help within a RAG pipeline.


We will use spaCy to break our text into sentences since it's likely a bit more robust than just using `text.split(". ")`. 

In [10]:
from spacy.lang.en import English

nlp = English()

nlp.add_pipe("sentencizer")

for item in tqdm(pages_and_texts):
    item["sentences"] = list(nlp(item["text"]).sents)
    item["sentences"] = [str(sentence) for sentence in item["sentences"]]
    
    item["page_sentence_count_spacy"] = len(item["sentences"])

  0%|          | 0/21 [00:00<?, ?it/s]

In [11]:
random.sample(pages_and_texts, k=1)

[{'page_number': -34,
  'page_char_count': 3157,
  'page_word_count': 703,
  'page_sentence_count_raw': 62,
  'page_token_count': 789.25,
  'text': 'offence  singly or jointly with any other person, commits that offence.  Illustrations      (a) A and B agree to murder Z by severally and at different times giving him small  doses of poison. A and B administer the poison according to the agreement with intent  to murder Z. Z dies from the effects of the several doses of poison so administered to  him. Here A and B intentionally co-operate in the commission of murder and as each of  them does an act by which the death is caused, they are both guilty of the offence  though their acts are separate.    (b) A and B are joint jailors, and as such, have the charge of Z, a prisoner, alternately  for six hours at a time. A and B, intending to cause Z\'s death, knowingly co-operate in  causing that effect by illegally omitting, each during the time of his attendance, to  furnish Z with food suppli

In [12]:
df = pd.DataFrame(pages_and_texts)
df

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text,sentences,page_sentence_count_spacy
0,-41,1967,503,14,491.75,"1 THE PENAL CODE, 1860 (ACT NO. XLV OF 1860...","[1 THE PENAL CODE, 1860 (ACT NO., XLV OF 18...",16
1,-40,2697,647,25,674.25,"(b) B, a European British subject, commits a m...","[(b) B, a European British subject, commits a ...",21
2,-39,2150,575,25,537.50,“Person” 11. The word “person” includes an...,"[“Person” 11., The word “person” includes ...",22
3,-38,3007,594,51,751.75,"“Court of justice” 20. The words ""Court of...","[“Court of justice” 20., The words ""Court ...",7
4,-37,2533,618,16,633.25,A Municipal Commissioner is a public servant. ...,[A Municipal Commissioner is a public servant....,18
5,-36,2625,556,33,656.25,"Property in possession of wife, clerk or ser...","[Property in possession of wife, clerk or se...",19
6,-35,2609,683,25,652.25,the signature. “Valuab...,"[the signature., “Valu...",20
7,-34,3157,703,62,789.25,offence singly or jointly with any other pers...,[offence singly or jointly with any other per...,24
8,-33,2373,691,32,593.25,"222, 223, 224, 225, 327, 328, 329, 330, 331, 3...","[222, 223, 224, 225, 327, 328, 329, 330, 331, ...",24
9,-32,2636,606,42,659.00,“Good faith” 52. Nothing is said to be don...,"[“Good faith” 52., Nothing is said to be d...",13


In [13]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy
count,21.0,21.00,21.00,21.00,21.00,21.00
mean,-31.0,2818.05,640.76,38.24,704.51,14.62
std,6.2,610.33,126.45,15.41,152.58,5.98
min,-41.0,958.00,191.00,14.00,239.50,4.00
25%,-36.0,2609.00,599.00,25.00,652.25,10.00
50%,-31.0,2968.00,683.00,41.00,742.00,15.00
75%,-26.0,3204.00,711.00,51.00,801.00,19.00
max,-21.0,3572.00,766.00,62.00,893.00,24.00


### Chunking our sentences together
Why do we do this?

1. Easier to manage similar sized chunks of text.
2. Don't overload the embedding models capacity for tokens (e.g. if an embedding model has a capacity of 384 tokens, there could be information loss if you try to embed a sequence of 400+ tokens).
3. Our LLM context window (the amount of tokens an LLM can take in) may be limited and requires compute power so we want to make sure we're using it as well as possible.

In [14]:
chunk_size = 10
def split_list(input_list: list[str], 
               slice_size: int=chunk_size) -> list[list[str]]:

    return [input_list[i:i+slice_size] for i in range(0, len(input_list), slice_size)]

for item in tqdm(pages_and_texts):
    item["sentence_chunks"] = split_list(input_list=item["sentences"],
                                         slice_size=chunk_size)
    item["num_chunks"] = len(item["sentence_chunks"])

  0%|          | 0/21 [00:00<?, ?it/s]

In [15]:
random.sample(pages_and_texts,k=1)

[{'page_number': -28,
  'page_char_count': 3572,
  'page_word_count': 733,
  'page_sentence_count_raw': 51,
  'page_token_count': 893.0,
  'text': 'servant  by such bidding, shall be punished with imprisonment of either description for a term which  may extend to one month, or with fine which may extend to two hundred taka, or with both.                       Obstructing public  servant in  discharge of  public functions     186. Whoever voluntarily obstructs any public servant in the discharge of his public functions,  shall be punished with imprisonment of either description for a term which may extend to  three months, or with fine which may extend to five hundred taka, or with both.                       Omission to assist  public servant  when bound by  law to give  assistance     187. Whoever, being bound by law to render or furnish assistance to any public servant in  the execution of his public duty, intentionally omits to give such assistance, shall be punished  with simple im

In [16]:
df = pd.DataFrame(pages_and_texts)
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,page_sentence_count_spacy,num_chunks
count,21.0,21.00,21.00,21.00,21.00,21.00,21.00
mean,-31.0,2818.05,640.76,38.24,704.51,14.62,1.86
std,6.2,610.33,126.45,15.41,152.58,5.98,0.73
min,-41.0,958.00,191.00,14.00,239.50,4.00,1.00
25%,-36.0,2609.00,599.00,25.00,652.25,10.00,1.00
50%,-31.0,2968.00,683.00,41.00,742.00,15.00,2.00
75%,-26.0,3204.00,711.00,51.00,801.00,19.00,2.00
max,-21.0,3572.00,766.00,62.00,893.00,24.00,3.00


### Splitting each chunk into its own item


In [17]:
import re

pages_and_chunks = []
for item in tqdm(pages_and_texts):
    for sentence_chunk in item["sentence_chunks"]:
        chunk_dict = {}
        chunk_dict["page_number"] = item["page_number"]
        
        # Join the sentences together into a paragraph-like structure, aka a chunk (so they are a single string)
        joined_sentence_chunk = "".join(sentence_chunk).replace("  "," ").strip()
        joined_sentence_chunk = re.sub(r'\.(A-Z)', r'. \1', joined_sentence_chunk) # convert ".A"to ". A"(only for capital letter)
        chunk_dict["sentence_chunk"] = joined_sentence_chunk
        
        chunk_dict["chunk_char_count"] = len(joined_sentence_chunk)
        chunk_dict["chunk_word_count"] = len([word for word in joined_sentence_chunk.split(" ")])
        chunk_dict["chunk_token_count"] = len(joined_sentence_chunk) / 4
        
        pages_and_chunks.append(chunk_dict)
    
len(pages_and_chunks)       

  0%|          | 0/21 [00:00<?, ?it/s]

39

In [18]:
random.sample(pages_and_chunks, k=1)

[{'page_number': -26,
  'sentence_chunk': 'Malignant act likely to spread infection of disease dangerous to life   270.Whoever malignantly does any act which is, and which he knows or has reason to believe to be, likely to spread the infection of any disease dangerous to life, shall be punished with imprisonment of either description for a term which may extend to two years, or with fine, or with both.            Disobedience to quarantine rule   271.Whoever knowingly disobeys any rule made and promulgated by the Government for putting any vessel into a state of quarantine, or for regulating the intercourse of vessels in a state of quarantine with the shore or with other vessels, or for regulating the intercourse between places where an infectious disease prevails and other places, shall be punished with imprisonment of either description for a term which may extend to six months, or with fine, or with both.            Adulteration of food  272.Whoever adulterates any article of food o

In [19]:
df = pd.DataFrame(pages_and_chunks)
df.describe().round(2)

,page_number,chunk_char_count,chunk_word_count,chunk_token_count
count,39.00,39.00,39.00,39.00
mean,-32.21,1449.28,277.36,362.32
std,5.98,917.48,166.81,229.37
min,-41.00,111.00,22.00,27.75
25%,-37.00,743.50,148.00,185.88
50%,-33.00,1204.00,240.00,301.00
75%,-27.50,1998.00,390.50,499.50
max,-21.00,3366.00,620.00,841.50


In [20]:
 df.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count
0,-41,"1 THE PENAL CODE, 1860 (ACT NO.XLV OF 1860). ...",1289,279,322.25
1,-41,(3) [Omitted by section 3 and 2nd Schedule of ...,545,92,136.25
2,-40,"(b) B, a European British subject, commits a m...",1251,240,312.75
3,-40,"Illustrations (a) The sections in this Code,...",1196,248,299.00
4,-40,The word “man” denotes a male human being of a...,111,22,27.75


In [22]:
min_token_length = 30
for row in df[df["chunk_token_count"] <= min_token_length].sample(5).iterrows():
    print(f'Chunk token count : {row[1]["chunk_token_count"]} | Text: {row[1]["sentence_chunk"]}')

ValueError: Cannot take a larger sample than population when 'replace=False'

In [23]:
#filtering rows with token under 30
pages_and_chunks_over_min_token_len = df[df["chunk_token_count"] > min_token_length].to_dict(orient="records")
pages_and_chunks_over_min_token_len[:2]

[{'page_number': -41,
  'sentence_chunk': '1 THE PENAL CODE, 1860  (ACT NO.XLV OF 1860).    [6th October, 1860]               CHAPTER I  INTRODUCTION       Preamble   WHEREAS it is expedient to provide a general Penal Code for Bangladesh; It is enacted as follows:-             Title and extent of operation of the Code   1.This Act shall be called the 2[ Penal Code], and shall take effect throughout Bangladesh.            Punishment of offences committed within Bangladesh   2.Every person shall be liable to punishment under this Code and not otherwise for every act or omission contrary to the provisions thereof, of which he shall be guilty within Bangladesh.            Punishment of offences committed beyond, but which by law may be tried within Bangladesh   3.Any person liable, by any Bangladesh Law, to be tried for an offence committed beyond Bangladesh shall be dealt with according to the provisions of this Code for any act committed beyond Bangladesh in the same manner as if such ac

In [24]:
random.sample(pages_and_chunks_over_min_token_len, k=1)

[{'page_number': -41,
  'sentence_chunk': '1 THE PENAL CODE, 1860  (ACT NO.XLV OF 1860).    [6th October, 1860]               CHAPTER I  INTRODUCTION       Preamble   WHEREAS it is expedient to provide a general Penal Code for Bangladesh; It is enacted as follows:-             Title and extent of operation of the Code   1.This Act shall be called the 2[ Penal Code], and shall take effect throughout Bangladesh.            Punishment of offences committed within Bangladesh   2.Every person shall be liable to punishment under this Code and not otherwise for every act or omission contrary to the provisions thereof, of which he shall be guilty within Bangladesh.            Punishment of offences committed beyond, but which by law may be tried within Bangladesh   3.Any person liable, by any Bangladesh Law, to be tried for an offence committed beyond Bangladesh shall be dealt with according to the provisions of this Code for any act committed beyond Bangladesh in the same manner as if such ac

### Embedding our text chunks

Embeddings of text will mean that similar meaning texts have similar numerical representation.


Our goal is to turn each of our chunks into a numerical representation (an embedding vector, where a vector is a sequence of numbers arranged in order).

We'll use our computers to find patterns in the embeddings and then we can use their text mappings to further our understanding.

We'll use the [`sentence-transformers`](https://www.sbert.net/docs/installation.html) library which contains many pre-trained embedding models.

Specifically, we'll get the `all-mpnet-base-v2` model (you can see the model's intended use on the [Hugging Face model card](https://huggingface.co/sentence-transformers/all-mpnet-base-v2#intended-uses)).

In [25]:
!pip install sentence-transformers # for embedding models

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 11.2 MB/s eta 0:00:00


In [26]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(model_name_or_path="all-mpnet-base-v2",
                                      device="cuda")


for item in tqdm(pages_and_chunks_over_min_token_len):
    item["embedding"] = embedding_model.encode(item["sentence_chunk"])


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [27]:
pages_and_chunks_over_min_token_len[0]["embedding"].shape

(768,)

Our embedding has a shape of `(768,)` meaning it's a vector of 768 numbers which represent our text in high-dimensional space.

In [28]:
# Turn text chunks into a single list
text_chunks = [item["sentence_chunk"] for item in pages_and_chunks_over_min_token_len]
text_chunks[9]

'Losing wrongfully   A person is said to gain wrongfully when such person retains wrongfully, as well as when such person acquires wrongfully.A person is said to loss wrongfully when such person is wrongfully kept out of any property, as well as when such person is wrongfully deprived of property.            “Dishonestly”   24.Whoever does anything with the intention of causing wrongful gain to one person or wrongful loss to another person, is said to do that thing "dishonestly".            “Fraudulently"   25.A person is said to do a thing fraudulently if he does that thing with intent to defraud but not otherwise.            “Reason to believe”  26.A person is said to have "reason to believe" a thing if he has sufficient cause to believe that thing but not otherwise.'

In [29]:
len(text_chunks)

38

In [30]:
text_chunk_embeddings = embedding_model.encode(text_chunks,
                                               batch_size=16, # Embed all texts in batches
                                               convert_to_tensor=True)
text_chunk_embeddings[0]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

tensor([ 4.0765e-02, -6.9201e-02,  2.3881e-02, -5.3142e-02, -4.3559e-02,
         4.9251e-02,  2.5954e-02,  2.1812e-02,  5.9380e-02,  4.0764e-02,
         3.2882e-03, -3.7364e-02,  1.4155e-02,  1.2345e-02,  6.3931e-02,
         4.1709e-02,  4.5600e-03,  5.8557e-03,  5.7257e-03,  2.6336e-02,
         1.8689e-02,  1.7316e-02,  7.4427e-03,  5.4950e-03,  3.2433e-02,
         2.4070e-02,  4.5923e-03,  1.2356e-02, -5.4996e-02, -1.1292e-02,
         5.2693e-02,  2.2630e-02,  4.0594e-02, -1.7483e-02,  1.6937e-06,
        -2.6182e-02, -1.7500e-02, -3.9062e-02, -4.1819e-02,  3.6031e-02,
        -3.9795e-02, -3.1730e-04,  1.1715e-02, -1.5238e-02, -3.9628e-02,
        -4.7980e-02,  1.1799e-02,  6.3447e-02,  9.3251e-03,  1.8541e-02,
         9.4109e-03, -8.3883e-02, -5.5643e-02, -4.0032e-02,  5.0022e-02,
         1.4523e-02, -7.0000e-03,  5.1976e-02,  4.4669e-02,  4.6007e-02,
         1.5093e-03, -1.6249e-02,  3.0002e-02, -2.5551e-02, -6.9359e-03,
        -7.3252e-02, -1.2194e-02,  9.0582e-03,  5.1

In [31]:
#Saving embedding to file
text_chunks_and_embeddings_df = pd.DataFrame(pages_and_chunks_over_min_token_len)
save_path = "text_chunks_and_embeddings_df.csv"
text_chunks_and_embeddings_df.to_csv(save_path, index=False)

In [32]:
# Import saved file and view
text_chunks_and_embeddings_df_load = pd.read_csv(save_path)
text_chunks_and_embeddings_df_load.head()

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-41,"1 THE PENAL CODE, 1860 (ACT NO.XLV OF 1860). ...",1289,279,322.25,[ 4.07653823e-02 -6.92013055e-02 2.38808561e-...
1,-41,(3) [Omitted by section 3 and 2nd Schedule of ...,545,92,136.25,[ 4.89617810e-02 -5.62559925e-02 3.61934397e-...
2,-40,"(b) B, a European British subject, commits a m...",1251,240,312.75,[ 3.32908295e-02 -5.96009120e-02 2.72631161e-...
3,-40,"Illustrations (a) The sections in this Code,...",1196,248,299.00,[ 3.59783992e-02 -5.08492589e-02 2.00839378e-...
4,-39,“Person” 11.The word “person” includes any C...,628,143,157.00,[ 3.67720313e-02 -2.46235430e-02 3.37098632e-...


# RAG - Search and Answer

### Similarity search
Similarity search or semantic search or vector search is the idea of searching on *semantic*.

With keyword search, you are trying to match the string "apple" with the string "apple".

Whereas with similarity/semantic search, you may want to search "macronutrients functions".
And get back results that don't necessarily contain the words "macronutrients functions" but get back pieces of text that match that meaning.


In [33]:
import torch
import numpy as np
device = "cuda" if torch.cuda.is_available() else "cpu"

text_chunks_and_embedding_df = pd.read_csv(save_path)
#convert embedding to array (it got converted to string when it saved)
text_chunks_and_embedding_df["embedding"] = text_chunks_and_embedding_df["embedding"].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))

#converting embedding into torch tensor
embeddings = torch.tensor(np.stack(text_chunks_and_embedding_df["embedding"].tolist(), axis=0), dtype=torch.float32).to(device)
# Convert texts and embedding df to list of dicts
pages_and_chunks = text = text_chunks_and_embedding_df.to_dict(orient="records")

text_chunks_and_embeddings_df

,page_number,sentence_chunk,chunk_char_count,chunk_word_count,chunk_token_count,embedding
0,-41,"1 THE PENAL CODE, 1860 (ACT NO.XLV OF 1860). ...",1289,279,322.25,"[0.040765382, -0.069201306, 0.023880856, -0.05..."
1,-41,(3) [Omitted by section 3 and 2nd Schedule of ...,545,92,136.25,"[0.04896178, -0.056255993, 0.003619344, -0.024..."
2,-40,"(b) B, a European British subject, commits a m...",1251,240,312.75,"[0.03329083, -0.059600912, 0.027263116, -0.020..."
3,-40,"Illustrations (a) The sections in this Code,...",1196,248,299.00,"[0.0359784, -0.05084926, 0.020083938, -0.00351..."
4,-39,“Person” 11.The word “person” includes any C...,628,143,157.00,"[0.03677203, -0.024623543, 0.0033709863, 0.015..."
5,-39,[Repealed] 16. [Repealed by the Government o...,1119,225,279.75,"[0.043088928, -0.011165836, 0.006733916, -0.01..."
6,-39,(c) [Repealed by the Federal Laws (Revision an...,237,43,59.25,"[0.030149743, -0.054981977, 0.039542355, 0.001..."
7,-38,"“Court of justice” 20.The words ""Court of Ju...",2934,521,733.50,"[0.066430435, -0.053875513, 0.02053645, 0.0006..."
8,-37,A Municipal Commissioner is a public servant. ...,1602,302,400.50,"[0.00032928947, 0.052225422, 0.042397227, 0.08..."
9,-37,Losing wrongfully A person is said to gain w...,778,164,194.50,"[-0.019034104, 0.06002082, 0.009006251, 0.0687..."


In [34]:
embeddings.shape

torch.Size([38, 768])

Retrival is done by following steps:
1. Define a query string.
2. Turn the query string in an embedding with same model we used to embed our text chunks.
3. Perform a [dot product](https://pytorch.org/docs/stable/generated/torch.dot.html) or [cosine similarity](https://en.wikipedia.org/wiki/Cosine_similarity) function between the text embeddings and the query embedding to get similarity scores.
4. Sort the results from step 3 in descending order (a higher score means more similarity in the eyes of the model) and use these values to inspect the texts. 

In [35]:
from sentence_transformers import util

query = "macronutrients functions"
print(f"Query : {query}")

query_embedding = embedding_model.encode(query, convert_to_tensor=True).to("cuda")

dot_scores = util.dot_score(query_embedding, embeddings)[0]

top_results = torch.topk(dot_scores, k=5)
top_results

Query : macronutrients functions


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

torch.return_types.topk(
values=tensor([0.0886, 0.0865, 0.0830, 0.0767, 0.0709], device='cuda:0'),
indices=tensor([31, 19, 14,  7,  4], device='cuda:0'))

In [36]:
for score, idx in zip(top_results[0], top_results[1]):
    print(f"Score: {score:.4f}")
    print("Text")
    print(pages_and_chunks[idx]["sentence_chunk"])
    print("\n\n")


Score: 0.0886
Text
or drink intended for sale noxious as food or drink, intending to sell such article as food or drink, or knowing it to be likely that the same will be sold as food or drink, shall be punished with imprisonment of either description for a term which may extend to six months, or with fine which may extend to one thousand taka, or with both.            Sale of noxious food or drink   273.Whoever sells, or offers or exposes for sale, as food or drink, any article which has been rendered or has become noxious, or is in a state unfit for food or drink, knowing or having reason to believe that the same is noxious as food or drink, shall be punished with imprisonment of either description for a term which may extend to six months, or with fine which may extend to one thousand taka, or with both.            Adulteration of drugs   274.Whoever adulterates any drug or medical preparation in such a manner as to lessen the efficacy or change the operation of such drug or medical 

In [37]:
def retrieve_relevant_resources(query: str, n_resources_to_return: int=5):
    
    query_embedding = embedding_model.encode(query, convert_to_tensor=True).to("cuda")

    dot_scores = util.dot_score(query_embedding, embeddings)[0]

    scores, indices = torch.topk(dot_scores, k=n_resources_to_return)
    
    return scores, indices



In [38]:
retrieve_relevant_resources(query)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(tensor([0.0886, 0.0865, 0.0830, 0.0767, 0.0709], device='cuda:0'),
 tensor([31, 19, 14,  7,  4], device='cuda:0'))

In [39]:
def print_top_results_and_scores(query: str, n_resources_to_return: int=5):
    """
    Takes a query, retrieves most relevant resources and prints them out in descending order.
    """
    scores, indices = retrieve_relevant_resources(query, n_resources_to_return=n_resources_to_return)
    for score, idx in zip(scores, indices):
        print(f"Score: {score:.4f}")
        print("Text")
        print(pages_and_chunks[idx]["sentence_chunk"])
        print("\n\n")

In [40]:
print_top_results_and_scores(query)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Score: 0.0886
Text
or drink intended for sale noxious as food or drink, intending to sell such article as food or drink, or knowing it to be likely that the same will be sold as food or drink, shall be punished with imprisonment of either description for a term which may extend to six months, or with fine which may extend to one thousand taka, or with both.            Sale of noxious food or drink   273.Whoever sells, or offers or exposes for sale, as food or drink, any article which has been rendered or has become noxious, or is in a state unfit for food or drink, knowing or having reason to believe that the same is noxious as food or drink, shall be punished with imprisonment of either description for a term which may extend to six months, or with fine which may extend to one thousand taka, or with both.            Adulteration of drugs   274.Whoever adulterates any drug or medical preparation in such a manner as to lessen the efficacy or change the operation of such drug or medical 

# Installing Gemma-2b
We will be using Gemma_instruct_2b for this.

In [41]:
!pip install -q -U keras-nlp
!pip install -q -U keras>=3

import os

os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"

/opt/conda/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [42]:
import keras
import keras_nlp
gemma_lm = keras_nlp.models.GemmaCausalLM.from_preset("gemma_instruct_2b_en") 

normalizer.cc(51) LOG(INFO) precompiled_charsmap is empty. use identity normalization.


In [43]:
input_text = "Is it legal for a person under the age of 9 to be held criminally liable according to penal code?use reference"

outputs = gemma_lm.generate(input_text, max_length=100)
print(outputs)

I0000 00:00:1733119504.743222      30 service.cc:145] XLA service 0x5c6a587d8ed0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1733119504.743296      30 service.cc:153]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1733119511.353147      30 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Is it legal for a person under the age of 9 to be held criminally liable according to penal code?use reference to relevant legal sources.

Sure, here's the answer to your question:

Whether a person under the age of 9 can be held criminally liable under the penal code depends on several factors, including the specific jurisdiction's laws and the nature of the charge.

**Relevant Legal Sources:**

* **Juvenile Delinquency Act (JDA)**


In [44]:
input_text = "is it illegal to drive without driving licsence,accourding to the penal code 1860?use artical, if you dont find it say no"

outputs = gemma_lm.generate(input_text, max_length=500)
print(outputs)

is it illegal to drive without driving licsence,accourding to the penal code 1860?use artical, if you dont find it say no.

Sure, here's the answer:

No, the penal code 1860 does not prohibit driving without a driver's license.


In [45]:
input_text = "Is it legal for a person to act in self-defense if their life is threatened accourding to the penal code 1860?use referance, if you dont find it say no"

outputs = gemma_lm.generate(input_text, max_length=500)
print(outputs)

Is it legal for a person to act in self-defense if their life is threatened accourding to the penal code 1860?use referance, if you dont find it say no.

Sure, here's a summary of the relevant portion of the penal code 1860 and my answer:

**Penal Code 1860, Section 24:**

> "A person who, in reasonable fear of imminent deadly or grievous harm, uses any means to defend himself or his property, or to prevent or repel an imminent violent attack, shall not be prosecuted for the act of self-defense."

**Answer:**

No, the Penal Code 1860 does not authorize a person to act in self-defense if their life is threatened. Self-defense is only permitted when the use of deadly force is absolutely necessary and there is no other reasonable means to protect oneself from imminent deadly or grievous harm.


In [ ]:
input_text = "Is it legal for a person to act in self-defense if their life is threatened accourding to the penal code 1860?use referance, if you dont find it say no"

outputs = gemma_lm.generate(input_text, max_length=500)
print(outputs)